# Analyse descriptive microstructure — Trades index futures SXE / DAX

Notebook réutilisable pour faire une **pré-analyse descriptive complète** avant Hasbrouck VAR, Hawkes, modèles de market impact ou modèles de toxicité.

Colonnes attendues :

| Colonne | Description |
|---|---|
| `timestamp` | timestamp du trade, datetime, secondes, millisecondes ou Unix timestamp |
| `price` | prix du trade |
| `qté` | quantité exécutée |
| `sens` | sens agressif : `+1` achat agressif, `-1` vente agressive |
| `mid` | mid-price au moment du trade |
| `bid` | meilleur bid au moment du trade |
| `ask` | meilleur ask au moment du trade |

Optionnel : `instrument` pour comparer DAX/SXE.

Objectifs : qualité de données, saisonnalité intraday, volumes, spreads, returns, imbalance, autocorrélation, runs/bursts, durations, realized volatility, et premier market impact descriptif.

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional, Dict, List, Tuple

try:
    from statsmodels.tsa.stattools import acf
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 1. Configuration

Adapte les noms de colonnes, tick sizes, fréquences de bucket et horaires de session.

La tick size sert à convertir les variations de prix en ticks :

$$
r_t^{ticks}=
rac{mid_t-mid_{t-1}}{tick\_size}
$$

In [ ]:
CONFIG = {
    "timestamp_col": "timestamp",
    "price_col": "price",
    "volume_col": "qté",
    "side_col": "sens",
    "mid_col": "mid",
    "bid_col": "bid",
    "ask_col": "ask",
    "instrument_col": "instrument",  # optionnel

    "default_tick_size": 0.5,
    "tick_size_by_instrument": {
        "DAX": 0.5,
        "FDAX": 0.5,
        "FDXM": 0.5,
        "SXE": 1.0,
        "FESX": 1.0,
    },

    "bucket_fast": "1s",
    "bucket_main": "1min",
    "bucket_slow": "5min",
    "impact_horizons_seconds": [1, 5, 10, 30, 60, 300],

    "timezone": None,
    "session_start": "08:00:00",
    "session_end": "22:00:00",
}

## 2. Charger les données

Renseigne `DATA_PATH`, ou charge directement ton DataFrame dans une variable `df` avant d'exécuter la suite.

In [ ]:
DATA_PATH = None  # ex: "/mnt/data/trades.csv" ou "/mnt/data/trades.parquet"

if DATA_PATH is not None:
    path = Path(DATA_PATH)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        raise ValueError("Format non supporté : utilise CSV ou Parquet.")
    print(df.shape)
    display(df.head())
else:
    print("DATA_PATH=None. Si ton DataFrame s'appelle déjà df, continue.")

## 3. Fonctions utilitaires

In [ ]:
def require_columns(df: pd.DataFrame, required: List[str]) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")


def convert_timestamp_series(s: pd.Series, timezone: Optional[str] = None) -> pd.Series:
    if np.issubdtype(s.dtype, np.datetime64):
        out = pd.to_datetime(s)
    elif np.issubdtype(s.dtype, np.number):
        med = s.dropna().median()
        if med > 1e12:
            out = pd.to_datetime(s, unit="ms", errors="coerce")
        elif med > 1e9:
            out = pd.to_datetime(s, unit="s", errors="coerce")
        else:
            out = pd.Timestamp("2000-01-01") + pd.to_timedelta(s, unit="s")
    else:
        out = pd.to_datetime(s, errors="coerce")

    if timezone is not None:
        try:
            if out.dt.tz is None:
                out = out.dt.tz_localize(timezone)
            else:
                out = out.dt.tz_convert(timezone)
        except Exception:
            pass
    return out


def map_side_to_pm1(s: pd.Series) -> pd.Series:
    if np.issubdtype(s.dtype, np.number):
        return pd.to_numeric(s, errors="coerce")
    mapping = {
        "BUY": 1, "Buy": 1, "buy": 1, "B": 1, "b": 1,
        "ACHAT": 1, "Achat": 1, "achat": 1,
        "ASK": 1, "Ask": 1, "ask": 1,
        "SELL": -1, "Sell": -1, "sell": -1, "S": -1, "s": -1,
        "VENTE": -1, "Vente": -1, "vente": -1,
        "BID": -1, "Bid": -1, "bid": -1,
    }
    return s.map(mapping)


def get_tick_size_for_rows(df: pd.DataFrame, config: Dict) -> pd.Series:
    instr_col = config["instrument_col"]
    default_tick = config["default_tick_size"]
    tick_map = config["tick_size_by_instrument"]
    if instr_col in df.columns:
        return df[instr_col].map(tick_map).fillna(default_tick).astype(float)
    return pd.Series(default_tick, index=df.index, dtype=float)


def clean_microstructure_df(df: pd.DataFrame, config: Dict = CONFIG) -> pd.DataFrame:
    d = df.copy()
    required = [config["timestamp_col"], config["price_col"], config["volume_col"], config["side_col"], config["mid_col"], config["bid_col"], config["ask_col"]]
    require_columns(d, required)

    d[config["timestamp_col"]] = convert_timestamp_series(d[config["timestamp_col"]], config.get("timezone"))
    d = d.dropna(subset=[config["timestamp_col"]])
    d = d.sort_values(config["timestamp_col"]).set_index(config["timestamp_col"])

    for c in [config["price_col"], config["volume_col"], config["mid_col"], config["bid_col"], config["ask_col"]]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d[config["side_col"]] = map_side_to_pm1(d[config["side_col"]])

    before = len(d)
    d = d.dropna(subset=[config["price_col"], config["volume_col"], config["side_col"], config["mid_col"], config["bid_col"], config["ask_col"]])
    d = d[d[config["volume_col"]] > 0]
    d = d[d[config["side_col"]].isin([-1, 1])]
    d = d[d[config["ask_col"]] >= d[config["bid_col"]]]
    after = len(d)

    d["tick_size"] = get_tick_size_for_rows(d.reset_index(), config).values
    d["spread"] = d[config["ask_col"]] - d[config["bid_col"]]
    d["spread_ticks"] = d["spread"] / d["tick_size"]
    d["mid_return"] = d[config["mid_col"]].diff()
    d["mid_return_ticks"] = d["mid_return"] / d["tick_size"]
    d["trade_return"] = d[config["price_col"]].diff()
    d["trade_return_ticks"] = d["trade_return"] / d["tick_size"]

    d["signed_qty"] = d[config["side_col"]] * d[config["volume_col"]]
    d["signed_log_qty"] = d[config["side_col"]] * np.log1p(d[config["volume_col"]])
    d["signed_sqrt_qty"] = d[config["side_col"]] * np.sqrt(d[config["volume_col"]])

    d["date"] = d.index.date
    d["hour"] = d.index.hour
    d["minute"] = d.index.minute
    d["tod_seconds"] = d.index.hour * 3600 + d.index.minute * 60 + d.index.second + d.index.microsecond / 1e6

    start = pd.to_timedelta(config["session_start"])
    end = pd.to_timedelta(config["session_end"])
    tod_td = pd.to_timedelta(d["tod_seconds"], unit="s")
    d["is_intraday_session"] = (tod_td >= start) & (tod_td <= end)

    print(f"Lignes initiales : {before:,}")
    print(f"Lignes conservées : {after:,}")
    print(f"Lignes retirées : {before-after:,}")
    return d


def summarize_missing(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": 100 * df.isna().mean(),
        "dtype": df.dtypes.astype(str)
    }).sort_values("missing_pct", ascending=False)


def basic_numeric_summary(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in df.columns]
    q = df[cols].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T
    desc = df[cols].describe().T
    return desc.join(q, rsuffix="_q")


def plot_hist(series: pd.Series, title: str, bins: int = 100, log_y: bool = False, clip_q: Optional[Tuple[float, float]] = None):
    x = series.dropna()
    if clip_q is not None and len(x) > 0:
        lo, hi = x.quantile(clip_q[0]), x.quantile(clip_q[1])
        x = x[(x >= lo) & (x <= hi)]
    plt.figure(figsize=(10, 5))
    plt.hist(x, bins=bins)
    if log_y:
        plt.yscale("log")
    plt.title(title)
    plt.xlabel(series.name if series.name else "value")
    plt.ylabel("count")
    plt.show()

## 4. Nettoyage et contrôle qualité

In [ ]:
try:
    df_clean = clean_microstructure_df(df, CONFIG)
    display(df_clean.head())
except NameError:
    raise NameError("Aucun DataFrame df trouvé. Charge tes données dans une variable df puis relance.")

missing_report = summarize_missing(df_clean)
display(missing_report)

print("Nombre de timestamps dupliqués :", df_clean.index.duplicated().sum())
print("Nombre total de trades :", len(df_clean))

bid_col, ask_col, mid_col = CONFIG["bid_col"], CONFIG["ask_col"], CONFIG["mid_col"]
print("Mid en dehors de [bid, ask] :", ((df_clean[mid_col] < df_clean[bid_col]) | (df_clean[mid_col] > df_clean[ask_col])).sum())
print("Spreads négatifs :", (df_clean["spread"] < 0).sum())
print("Spreads nuls :", (df_clean["spread"] == 0).sum())

## 5. Statistiques descriptives trade-level

In [ ]:
cols_summary = [CONFIG["price_col"], CONFIG["volume_col"], CONFIG["mid_col"], CONFIG["bid_col"], CONFIG["ask_col"],
                "spread", "spread_ticks", "mid_return_ticks", "trade_return_ticks", "signed_qty", "signed_log_qty"]
display(basic_numeric_summary(df_clean, cols_summary))

plot_hist(df_clean[CONFIG["volume_col"]], "Distribution des quantités par trade", bins=100, log_y=True, clip_q=(0.0, 0.99))
plot_hist(df_clean["spread_ticks"], "Distribution du spread en ticks", bins=100, log_y=True, clip_q=(0.0, 0.99))
plot_hist(df_clean["mid_return_ticks"], "Distribution des variations du mid en ticks", bins=100, log_y=True, clip_q=(0.01, 0.99))

## 6. Agrégation par buckets

Variables construites : volume, trade count, signed volume, imbalance, spread moyen, returns en ticks, realized volatility proxy.

In [ ]:
def aggregate_buckets(df: pd.DataFrame, bucket: str, config: Dict = CONFIG) -> pd.DataFrame:
    volume_col, side_col = config["volume_col"], config["side_col"]
    price_col, mid_col = config["price_col"], config["mid_col"]
    bid_col, ask_col = config["bid_col"], config["ask_col"]

    mid_b = df[mid_col].resample(bucket).last().ffill()
    price_b = df[price_col].resample(bucket).last().ffill()
    tick_b = df["tick_size"].resample(bucket).last().ffill()

    vol = df[volume_col].resample(bucket).sum().fillna(0)
    ntrades = df[volume_col].resample(bucket).count().fillna(0)
    signed_qty = df["signed_qty"].resample(bucket).sum().fillna(0)
    signed_log_qty = df["signed_log_qty"].resample(bucket).sum().fillna(0)

    buy_vol = df.loc[df[side_col] == 1, volume_col].resample(bucket).sum().reindex(vol.index).fillna(0)
    sell_vol = df.loc[df[side_col] == -1, volume_col].resample(bucket).sum().reindex(vol.index).fillna(0)
    imbalance = ((buy_vol - sell_vol) / (buy_vol + sell_vol)).replace([np.inf, -np.inf], np.nan).fillna(0)

    buy_count = df.loc[df[side_col] == 1, volume_col].resample(bucket).count().reindex(vol.index).fillna(0)
    sell_count = df.loc[df[side_col] == -1, volume_col].resample(bucket).count().reindex(vol.index).fillna(0)
    count_imbalance = ((buy_count - sell_count) / (buy_count + sell_count)).replace([np.inf, -np.inf], np.nan).fillna(0)

    out = pd.DataFrame({
        "mid": mid_b,
        "trade_price": price_b,
        "bid": df[bid_col].resample(bucket).last().ffill(),
        "ask": df[ask_col].resample(bucket).last().ffill(),
        "tick_size": tick_b,
        "r_mid_ticks": mid_b.diff() / tick_b,
        "r_trade_ticks": price_b.diff() / tick_b,
        "volume": vol,
        "trade_count": ntrades,
        "signed_qty": signed_qty,
        "signed_log_qty": signed_log_qty,
        "buy_volume": buy_vol,
        "sell_volume": sell_vol,
        "imbalance": imbalance,
        "buy_count": buy_count,
        "sell_count": sell_count,
        "count_imbalance": count_imbalance,
        "spread_ticks": df["spread_ticks"].resample(bucket).mean(),
    })
    out["abs_r_mid_ticks"] = out["r_mid_ticks"].abs()
    out["sq_r_mid_ticks"] = out["r_mid_ticks"] ** 2
    out["date"] = out.index.date
    out["hour"] = out.index.hour
    out["tod_seconds"] = out.index.hour * 3600 + out.index.minute * 60 + out.index.second
    return out.dropna(subset=["mid"])

bfast = aggregate_buckets(df_clean, CONFIG["bucket_fast"], CONFIG)
bmain = aggregate_buckets(df_clean, CONFIG["bucket_main"], CONFIG)
bslow = aggregate_buckets(df_clean, CONFIG["bucket_slow"], CONFIG)

print("bfast", bfast.shape, "bmain", bmain.shape, "bslow", bslow.shape)
display(bmain.head())

## 7. Profil intraday : volume, trades, spread, volatilité

In [ ]:
def intraday_profile(bucket_df: pd.DataFrame, value_cols: List[str]) -> pd.DataFrame:
    tmp = bucket_df.copy()
    tmp["time_of_day"] = tmp.index.strftime("%H:%M:%S")
    return tmp.groupby("time_of_day")[value_cols].mean()


def plot_intraday_profile(profile: pd.DataFrame, col: str, title: Optional[str] = None, max_points: int = 500):
    s = profile[col].dropna()
    if len(s) > max_points:
        step = int(np.ceil(len(s) / max_points))
        s = s.iloc[::step]
    plt.figure(figsize=(12, 5))
    plt.plot(range(len(s)), s.values)
    plt.title(title or f"Profil intraday — {col}")
    plt.xlabel("Time of day")
    plt.ylabel(col)
    if len(s) > 0:
        idx = np.linspace(0, len(s)-1, min(10, len(s))).astype(int)
        plt.xticks(idx, [s.index[i] for i in idx], rotation=45)
    plt.show()

profile_cols = ["volume", "trade_count", "spread_ticks", "abs_r_mid_ticks", "sq_r_mid_ticks", "imbalance"]
profile = intraday_profile(bmain, profile_cols)
display(profile.head())

for col in ["volume", "trade_count", "spread_ticks", "abs_r_mid_ticks", "sq_r_mid_ticks"]:
    plot_intraday_profile(profile, col)

## 8. Intraday vs extraday

In [ ]:
intra_stats = df_clean.groupby("is_intraday_session").agg(
    n_trades=(CONFIG["volume_col"], "count"),
    total_volume=(CONFIG["volume_col"], "sum"),
    avg_trade_size=(CONFIG["volume_col"], "mean"),
    median_trade_size=(CONFIG["volume_col"], "median"),
    avg_spread_ticks=("spread_ticks", "mean"),
    median_spread_ticks=("spread_ticks", "median"),
    avg_abs_mid_return_ticks=("mid_return_ticks", lambda x: x.abs().mean()),
)
intra_stats.index = intra_stats.index.map({True: "intraday_session", False: "extraday"})
display(intra_stats)

## 9. Signed order flow et imbalance

In [ ]:
plot_hist(bmain["imbalance"], "Distribution de l'imbalance volume par bucket", bins=100)
plot_hist(bmain["signed_qty"], "Distribution du signed volume par bucket", bins=100, log_y=True, clip_q=(0.01, 0.99))

plt.figure(figsize=(12, 5))
plt.plot(bmain.index, bmain["imbalance"].rolling(30, min_periods=5).mean())
plt.axhline(0, linestyle="--")
plt.title("Imbalance rolling moyenne")
plt.xlabel("time")
plt.ylabel("imbalance")
plt.show()

## 10. Autocorrélation du flux

In [ ]:
def plot_autocorrelation(series: pd.Series, lags: int = 100, title: str = "Autocorrelation"):
    x = series.dropna()
    if len(x) < lags + 5:
        print("Série trop courte pour cette autocorrélation.")
        return
    if HAS_STATSMODELS:
        vals = acf(x, nlags=lags, fft=True)
    else:
        vals = [1.0]
        xc = x - x.mean()
        denom = np.sum(xc ** 2)
        for k in range(1, lags + 1):
            vals.append(np.sum(xc.iloc[k:].values * xc.iloc[:-k].values) / denom)
        vals = np.array(vals)
    plt.figure(figsize=(10, 5))
    plt.stem(range(len(vals)), vals)
    plt.axhline(0, linestyle="--")
    plt.title(title)
    plt.xlabel("lag")
    plt.ylabel("ACF")
    plt.show()

plot_autocorrelation(bmain["signed_qty"], lags=60, title="ACF signed volume par bucket")
plot_autocorrelation(bmain["imbalance"], lags=60, title="ACF imbalance par bucket")

## 11. Runs et bursts directionnels

Un run est une séquence de trades consécutifs de même sens. C'est un proxy simple de méta-ordre avant Hawkes.

In [ ]:
def compute_runs(df: pd.DataFrame, config: Dict = CONFIG) -> pd.DataFrame:
    side_col, vol_col = config["side_col"], config["volume_col"]
    price_col, mid_col = config["price_col"], config["mid_col"]
    d = df.copy().sort_index()
    side = d[side_col].astype(int)
    d["run_id"] = (side != side.shift()).cumsum()
    runs = d.groupby("run_id").agg(
        start_time=(side_col, lambda x: x.index[0]),
        end_time=(side_col, lambda x: x.index[-1]),
        side=(side_col, "first"),
        n_trades=(side_col, "count"),
        total_qty=(vol_col, "sum"),
        avg_qty=(vol_col, "mean"),
        start_mid=(mid_col, "first"),
        end_mid=(mid_col, "last"),
        start_price=(price_col, "first"),
        end_price=(price_col, "last"),
    )
    runs["duration_seconds"] = (runs["end_time"] - runs["start_time"]).dt.total_seconds()
    runs["signed_total_qty"] = runs["side"] * runs["total_qty"]
    runs["mid_move_during_run"] = runs["end_mid"] - runs["start_mid"]
    runs["signed_mid_move_during_run"] = runs["side"] * runs["mid_move_during_run"]
    return runs

runs = compute_runs(df_clean, CONFIG)
display(runs.head())
display(runs[["n_trades", "total_qty", "duration_seconds", "signed_mid_move_during_run"]].describe())
plot_hist(runs["n_trades"], "Distribution des longueurs de runs", bins=80, log_y=True, clip_q=(0.0, 0.99))
plot_hist(runs["total_qty"], "Distribution du volume total des runs", bins=80, log_y=True, clip_q=(0.0, 0.99))
plot_hist(runs["duration_seconds"], "Distribution de la durée des runs", bins=80, log_y=True, clip_q=(0.0, 0.99))

## 12. Durées inter-trades

Pré-diagnostic pour point processes :

$$
	au_i=t_i-t_{i-1}
$$

In [ ]:
df_clean["interarrival_seconds"] = df_clean.index.to_series().diff().dt.total_seconds()
display(df_clean["interarrival_seconds"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
plot_hist(df_clean["interarrival_seconds"], "Distribution des durées inter-trades, secondes", bins=100, log_y=True, clip_q=(0.0, 0.99))

## 13. Volatilité réalisée descriptive

Sur bucket rapide $j$ :

$$
r_j=
rac{mid_j-mid_{j-1}}{tick\_size}
$$

Sur bucket plus lent :

$$
RV_t=\sum_{j\in t} r_j^2
$$

In [ ]:
def realized_vol_from_fast_buckets(bfast: pd.DataFrame, slow_bucket: str = "1min") -> pd.DataFrame:
    rv = bfast["r_mid_ticks"].pow(2).resample(slow_bucket).sum()
    absret = bfast["r_mid_ticks"].abs().resample(slow_bucket).sum()
    return pd.DataFrame({
        "RV_ticks2": rv,
        "AbsReturn_ticks": absret,
        "RealizedVol_ticks": np.sqrt(rv)
    })

rv_main = realized_vol_from_fast_buckets(bfast, CONFIG["bucket_main"])
display(rv_main.describe())

plt.figure(figsize=(12, 5))
plt.plot(rv_main.index, rv_main["RealizedVol_ticks"])
plt.title("Volatilité réalisée par bucket principal")
plt.xlabel("time")
plt.ylabel("Realized vol, ticks")
plt.show()

## 14. Market impact descriptif sans modèle

Event study simple :

$$
MI_i(	au)=sens_i\cdot 
rac{mid_{t_i+	au}-mid_{t_i}}{tick\_size}
$$

Ce n'est pas causal ; c'est un diagnostic avant modèles.

In [ ]:
def compute_trade_level_impact(df: pd.DataFrame, horizons_seconds: List[int], config: Dict = CONFIG, max_trades: Optional[int] = None) -> pd.DataFrame:
    d = df.copy().sort_index()
    if max_trades is not None and len(d) > max_trades:
        d = d.iloc[np.linspace(0, len(d)-1, max_trades).astype(int)]

    side_col, vol_col, mid_col = config["side_col"], config["volume_col"], config["mid_col"]
    base = d[[side_col, vol_col, mid_col, "tick_size", "spread_ticks", "signed_qty", "signed_log_qty"]].copy().reset_index()
    ts_col = base.columns[0]
    base = base.rename(columns={ts_col: "timestamp", mid_col: "mid_0"})

    mids = df[[mid_col]].copy().sort_index().reset_index()
    mids.columns = ["timestamp_future", "mid_future"]
    out = base.copy()

    for h in horizons_seconds:
        query = out[["timestamp"]].copy()
        query["timestamp_future"] = query["timestamp"] + pd.to_timedelta(h, unit="s")
        query = query[["timestamp_future"]]
        matched = pd.merge_asof(query.sort_values("timestamp_future"), mids.sort_values("timestamp_future"), on="timestamp_future", direction="forward")
        matched = matched.reindex(query.sort_values("timestamp_future").index)
        out[f"mid_{h}s"] = matched["mid_future"].values
        out[f"impact_{h}s_ticks"] = out[side_col] * (out[f"mid_{h}s"] - out["mid_0"]) / out["tick_size"]
    return out

impact_horizons = CONFIG["impact_horizons_seconds"]
impact_df = compute_trade_level_impact(df_clean, impact_horizons, CONFIG, max_trades=200_000)
impact_cols = [f"impact_{h}s_ticks" for h in impact_horizons]
display(impact_df.head())
display(impact_df[impact_cols].describe(percentiles=[0.05, 0.5, 0.95]))

mean_impact = pd.Series({h: impact_df[f"impact_{h}s_ticks"].mean() for h in impact_horizons})
median_impact = pd.Series({h: impact_df[f"impact_{h}s_ticks"].median() for h in impact_horizons})
plt.figure(figsize=(10, 5))
plt.plot(mean_impact.index, mean_impact.values, marker="o", label="mean")
plt.plot(median_impact.index, median_impact.values, marker="o", label="median")
plt.axhline(0, linestyle="--")
plt.xlabel("Horizon, seconds")
plt.ylabel("Signed impact, ticks")
plt.title("Market impact descriptif moyen")
plt.legend()
plt.show()

## 15. Impact par taille et par sens

In [ ]:
def impact_by_size_bins(impact_df: pd.DataFrame, horizon: int, n_bins: int = 10, config: Dict = CONFIG) -> pd.DataFrame:
    vol_col = config["volume_col"]
    col = f"impact_{horizon}s_ticks"
    d = impact_df[[vol_col, col]].dropna().copy()
    d["size_bin"] = pd.qcut(d[vol_col], q=n_bins, duplicates="drop")
    return d.groupby("size_bin").agg(
        n=(col, "count"),
        avg_qty=(vol_col, "mean"),
        median_qty=(vol_col, "median"),
        mean_impact=(col, "mean"),
        median_impact=(col, "median"),
    ).reset_index()

h0 = impact_horizons[min(2, len(impact_horizons)-1)]
impact_size = impact_by_size_bins(impact_df, horizon=h0, n_bins=10, config=CONFIG)
display(impact_size)
plt.figure(figsize=(10, 5))
plt.plot(impact_size["avg_qty"], impact_size["mean_impact"], marker="o")
plt.axhline(0, linestyle="--")
plt.xlabel("Average trade size in bin")
plt.ylabel(f"Mean signed impact at {h0}s, ticks")
plt.title("Impact descriptif par taille de trade")
plt.show()

side_col = CONFIG["side_col"]
impact_by_side = []
for h in impact_horizons:
    col = f"impact_{h}s_ticks"
    tmp = impact_df.groupby(side_col)[col].agg(["count", "mean", "median", "std"])
    tmp["horizon_s"] = h
    impact_by_side.append(tmp.reset_index())
impact_by_side = pd.concat(impact_by_side, ignore_index=True)
display(impact_by_side)

plt.figure(figsize=(10, 5))
for s, lab in [(1, "Buy aggressive"), (-1, "Sell aggressive")]:
    tmp = impact_by_side[impact_by_side[side_col] == s]
    plt.plot(tmp["horizon_s"], tmp["mean"], marker="o", label=lab)
plt.axhline(0, linestyle="--")
plt.xlabel("Horizon, seconds")
plt.ylabel("Mean signed impact, ticks")
plt.title("Impact descriptif par sens")
plt.legend()
plt.show()

## 16. Relation descriptive entre imbalance et rendements futurs

In [ ]:
def add_future_returns(bucket_df: pd.DataFrame, horizons_buckets: List[int]) -> pd.DataFrame:
    d = bucket_df.copy()
    for h in horizons_buckets:
        d[f"future_r_{h}b_ticks"] = (d["mid"].shift(-h) - d["mid"]) / d["tick_size"]
    return d

horizons_buckets = [1, 5, 10, 30]
b_pred = add_future_returns(bmain, horizons_buckets)

def corr_table(df: pd.DataFrame, xcols: List[str], ycols: List[str]) -> pd.DataFrame:
    rows = []
    for x in xcols:
        for y in ycols:
            tmp = df[[x, y]].dropna()
            rows.append({"x": x, "y": y, "corr": tmp[x].corr(tmp[y]), "n": len(tmp)})
    return pd.DataFrame(rows)

xcols = ["signed_qty", "signed_log_qty", "imbalance", "count_imbalance", "volume", "trade_count", "spread_ticks"]
ycols = [f"future_r_{h}b_ticks" for h in horizons_buckets]
ct = corr_table(b_pred, xcols, ycols)
display(ct.pivot(index="x", columns="y", values="corr"))

h = horizons_buckets[0]
plt.figure(figsize=(8, 6))
plt.scatter(b_pred["imbalance"], b_pred[f"future_r_{h}b_ticks"], alpha=0.2, s=10)
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("Imbalance")
plt.ylabel(f"Future return {h} bucket(s), ticks")
plt.title("Imbalance vs future return")
plt.show()

## 17. Comparaison par instrument, si la colonne existe

In [ ]:
instr_col = CONFIG["instrument_col"]
if instr_col in df_clean.columns:
    inst_stats = df_clean.groupby(instr_col).agg(
        n_trades=(CONFIG["volume_col"], "count"),
        total_volume=(CONFIG["volume_col"], "sum"),
        avg_qty=(CONFIG["volume_col"], "mean"),
        median_qty=(CONFIG["volume_col"], "median"),
        avg_spread_ticks=("spread_ticks", "mean"),
        median_spread_ticks=("spread_ticks", "median"),
        avg_abs_mid_return_ticks=("mid_return_ticks", lambda x: x.abs().mean()),
    )
    display(inst_stats)
    for inst, sub in df_clean.groupby(instr_col):
        print("Instrument:", inst)
        b_inst = aggregate_buckets(sub, CONFIG["bucket_main"], CONFIG)
        prof_inst = intraday_profile(b_inst, ["volume", "trade_count", "spread_ticks", "abs_r_mid_ticks"])
        plot_intraday_profile(prof_inst, "volume", title=f"Profil intraday volume — {inst}")
        plot_intraday_profile(prof_inst, "spread_ticks", title=f"Profil intraday spread ticks — {inst}")
else:
    print(f"Colonne {instr_col!r} absente : comparaison par instrument ignorée.")

## 18. Analyse par date / session

In [ ]:
daily_stats = df_clean.groupby("date").agg(
    n_trades=(CONFIG["volume_col"], "count"),
    total_volume=(CONFIG["volume_col"], "sum"),
    avg_qty=(CONFIG["volume_col"], "mean"),
    median_qty=(CONFIG["volume_col"], "median"),
    avg_spread_ticks=("spread_ticks", "mean"),
    median_spread_ticks=("spread_ticks", "median"),
    abs_mid_return_ticks=("mid_return_ticks", lambda x: x.abs().sum()),
    signed_volume=("signed_qty", "sum"),
)
daily_stats["daily_imbalance"] = daily_stats["signed_volume"] / daily_stats["total_volume"]
display(daily_stats.head())

daily_stats[["n_trades", "total_volume", "avg_spread_ticks", "abs_mid_return_ticks", "daily_imbalance"]].plot(figsize=(12, 8), subplots=True, title="Statistiques journalières")
plt.tight_layout()
plt.show()

## 19. Rapport synthétique automatique

In [ ]:
def make_summary_report(df_clean: pd.DataFrame, bmain: pd.DataFrame, impact_df: pd.DataFrame, impact_horizons: List[int], config: Dict = CONFIG) -> str:
    vol_col, side_col = config["volume_col"], config["side_col"]
    lines = []
    lines.append("RAPPORT DESCRIPTIF MICROSTRUCTURE")
    lines.append("=" * 50)
    lines.append(f"Nombre de trades : {len(df_clean):,}")
    lines.append(f"Période : {df_clean.index.min()} -> {df_clean.index.max()}")
    lines.append(f"Volume total : {df_clean[vol_col].sum():,.0f}")
    lines.append(f"Taille moyenne trade : {df_clean[vol_col].mean():.3f}")
    lines.append(f"Taille médiane trade : {df_clean[vol_col].median():.3f}")
    lines.append(f"Spread moyen, ticks : {df_clean['spread_ticks'].mean():.3f}")
    lines.append(f"Spread médian, ticks : {df_clean['spread_ticks'].median():.3f}")
    lines.append(f"Part buy agressif : {(df_clean[side_col].eq(1).mean()*100):.2f}%")
    lines.append(f"Part sell agressif : {(df_clean[side_col].eq(-1).mean()*100):.2f}%")
    lines.append(f"Imbalance volume globale : {df_clean['signed_qty'].sum()/df_clean[vol_col].sum():.4f}")
    lines.append("")
    lines.append("Impact descriptif moyen :")
    for h in impact_horizons:
        col = f"impact_{h}s_ticks"
        if col in impact_df.columns:
            lines.append(f"  Horizon {h}s : mean={impact_df[col].mean():.4f} ticks, median={impact_df[col].median():.4f} ticks")
    lines.append("")
    lines.append("Buckets :")
    lines.append(f"  Nombre buckets : {len(bmain):,}")
    lines.append(f"  Volume moyen par bucket : {bmain['volume'].mean():.3f}")
    lines.append(f"  Trades moyens par bucket : {bmain['trade_count'].mean():.3f}")
    lines.append(f"  Imbalance moyenne par bucket : {bmain['imbalance'].mean():.4f}")
    return "
".join(lines)

report = make_summary_report(df_clean, bmain, impact_df, impact_horizons, CONFIG)
print(report)

## 20. Export des tables utiles

Décommente les lignes pour exporter les tables préparées.

In [ ]:
EXPORT_DIR = Path("./exports_microstructure")
EXPORT_DIR.mkdir(exist_ok=True)

# df_clean.to_parquet(EXPORT_DIR / "trades_clean.parquet")
# bmain.to_parquet(EXPORT_DIR / f"buckets_{CONFIG['bucket_main']}.parquet")
# bfast.to_parquet(EXPORT_DIR / f"buckets_{CONFIG['bucket_fast']}.parquet")
# runs.to_parquet(EXPORT_DIR / "runs.parquet")
# impact_df.to_parquet(EXPORT_DIR / "impact_descriptif.parquet")
# daily_stats.to_csv(EXPORT_DIR / "daily_stats.csv")
# profile.to_csv(EXPORT_DIR / "intraday_profile.csv")

print("Dossier export prêt :", EXPORT_DIR.resolve())

## 21. Lecture des résultats

Avant de passer à Hasbrouck ou Hawkes :

- spreads souvent supérieurs à 1 tick : l'impact trade-only doit être interprété avec prudence ;
- imbalance fortement autocorrélée : flux persistant, méta-ordres probables ;
- runs longs fréquents : flux agressif non indépendant ;
- impact descriptif qui monte puis redescend : composante temporaire / reversal ;
- impact descriptif qui monte puis se stabilise : composante informationnelle possible ;
- DAX et SXE très différents : modèles séparés, normalisation par tick, volatilité et régime intraday.

Prochaines étapes :

1. VAR de Hasbrouck sur `r_mid_ticks` et `signed_log_qty`.
2. Hawkes buy/sell sur timestamps et sens.
3. Hawkes 4D : `BuyTrade`, `SellTrade`, `UpMove`, `DownMove`.
4. Ajout de spread, RV, TOD, imbalance comme covariates.
5. Modèles séparés open / midday / US overlap / close / extraday.